# 2 · Understand the image inputs

## Goal

Identify the pinned software inputs, image layout and packages **before** running a privileged build.

## Prerequisites

Complete the host preflight; all cells below are read-only.

In [ ]:
from pathlib import Path
import os, subprocess, sys
START = Path.cwd().resolve()
ROOT = next((p for p in (START, *START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()), None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+", " ".join(map(str, argv)), flush=True)
    return subprocess.run([str(a) for a in argv], cwd=ROOT, check=True)
print("Source:", ROOT)

## Action 1 · Read pinned versions

The container uses a Debian snapshot and `mkosi` package defined in `config/build.json`.

In [ ]:
import json
build_config=json.loads((ROOT/"config/build.json").read_text())
for key, value in sorted(build_config.items()):
    print(f"{key:24} {value}")

## Action 2 · Review server packages

These are mkosi package inputs, not a promise of the exact installed footprint.

In [ ]:
import configparser
def read_mkosi(name):
    parser=configparser.ConfigParser(interpolation=None)
    parser.read(ROOT/"images"/name/"mkosi.conf")
    return parser
server=read_mkosi("server")
installer=read_mkosi("installer")
for role,cfg in (("Server",server),("Installer",installer)):
    names=cfg.get("Content","Packages").split()
    print(f"{role} packages ({len(names)}):", ", ".join(names))

## Action 3 · Inspect partitions and boot command

The server contains an ESP and writable ext4 root. The UKI embeds the kernel and initrd.

In [ ]:
for path in sorted((ROOT/"images/server/mkosi.repart").glob("*.conf")):
    print("\n", path.name)
    print(path.read_text())
print("Kernel command line:",server.get("Content","KernelCommandLine"))

## Action 4 · Run static source checks

This executes the canonical checks without creating image artifacts.

In [ ]:
run("make","check")

## Check

You should know what is pinned, which packages go into each image, and why the disk has multiple partitions.

## Next

[Validate FreeIPA CA trust](03-trust.ipynb).